# Mise à jour du classeur de la pharmacie

Ce notebook **met à jour le classeur Excel** `excel/Pharma_Analyse.xlsx` à partir du dernier export de ventes.

Le classeur reste un fichier Excel, car la pharmacienne l'ouvre sans Python. C'est ce notebook qui le remplit.

**Mode d'emploi, à chaque nouvel export :**
1. Remplacer `data/Pharma_Ventes_Daily.csv` par le nouvel export.
2. Fermer le classeur s'il est ouvert dans Excel.
3. Cliquer sur **Run All** (Exécuter tout).
4. Ouvrir `excel/Pharma_Analyse.xlsx` : les ventes, les statistiques, les dates et la prévision du mois suivant sont à jour.

In [1]:
import re
import warnings
from copy import copy

import openpyxl
import pandas as pd
from openpyxl.styles import PatternFill

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

FICHIER_VENTES = "data/Pharma_Ventes_Daily.csv"
FICHIER_HORAIRE = "data/Pharma_Ventes_Hourly.csv"
FICHIER_CLASSEUR = "excel/Pharma_Analyse.xlsx"
N_MOIS_TEST = 21      # nombre de mois sur lesquels on teste la prévision
GAIN_MINIMUM = 2      # points d'erreur en moins exigés pour dire « prévisible »

GROUPES = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
NOMS = {
    "M01AB": "Anti-inflammatoires type diclofénac", "M01AE": "Anti-inflammatoires type ibuprofène",
    "N02BA": "Aspirine et dérivés", "N02BE": "Paracétamol", "N05B": "Anxiolytiques",
    "N05C": "Somnifères et sédatifs", "R03": "Asthme et BPCO", "R06": "Antihistaminiques (allergies)",
}
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
MOIS_FR = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet",
           "août", "septembre", "octobre", "novembre", "décembre"]

def en_francais(date):
    """2019-10-01 -> 'octobre 2019'"""
    return f"{MOIS_FR[date.month - 1]} {date.year}"

def pourcent(x):
    """13.04 -> '13,0 %'"""
    return f"{x:.1f} %".replace(".", ",")

## Étape 1 : lire les ventes du dernier export

In [2]:
jour = pd.read_csv(FICHIER_VENTES)
jour["datum"] = pd.to_datetime(jour["datum"], format="%m/%d/%Y")
jour = jour.sort_values("datum").reset_index(drop=True)
premier_jour, dernier_jour = jour["datum"].min(), jour["datum"].max()
print(f"{len(jour)} jours, du {premier_jour:%d/%m/%Y} au {dernier_jour:%d/%m/%Y}")

2106 jours, du 02/01/2014 au 08/10/2019


## Étape 2 : ventes par mois, en gardant seulement les mois complets

Un mois est complet si l'export contient tous ses jours. Le mois à prévoir est celui qui suit le dernier mois complet.

In [3]:
mois = jour.set_index("datum")[GROUPES].resample("MS").sum()
jours_par_mois = jour.set_index("datum").resample("MS").size()
mois = mois[jours_par_mois == jours_par_mois.index.days_in_month]

dernier_mois = mois.index.max()
mois_cible = dernier_mois + pd.DateOffset(months=1)
print(f"{len(mois)} mois complets, de {en_francais(mois.index.min())} à {en_francais(dernier_mois)}")
print(f"Mois à prévoir : {en_francais(mois_cible)}")

68 mois complets, de février 2014 à septembre 2019
Mois à prévoir : octobre 2019


## Étape 3 : tester les 4 méthodes de prévision

On se replace chacun des derniers mois, on prévoit avec uniquement ce qu'on savait avant, puis on compare au réel.
- 2 méthodes « naïves » : le mois dernier ; le même mois l'an dernier.
- 2 méthodes à nous : la moyenne des 3 derniers mois ; le niveau des 12 derniers mois × l'effet de saison du mois.

In [4]:
def naive_mois_dernier(historique, cible):
    return historique.iloc[-1]

def naive_annee_derniere(historique, cible):
    return historique.loc[cible - pd.DateOffset(years=1)]

def moyenne_3_mois(historique, cible):
    return historique.iloc[-3:].mean()

def saison_et_niveau(historique, cible):
    niveau = historique.iloc[-12:].mean()
    indice = (historique / historique.rolling(12, center=True).mean()).dropna()
    return niveau * indice[indice.index.month == cible.month].mean()

METHODES = {
    "Naïve : mois dernier": naive_mois_dernier,
    "Naïve : même mois an dernier": naive_annee_derniere,
    "Moyenne 3 derniers mois": moyenne_3_mois,
    "Saison x niveau": saison_et_niveau,
}
NAIVES = ["Naïve : mois dernier", "Naïve : même mois an dernier"]
MODELES = ["Moyenne 3 derniers mois", "Saison x niveau"]

mois_testes = mois.index[-N_MOIS_TEST:]
lignes = []
for cible in mois_testes:
    historique = mois[mois.index < cible]
    for nom, methode in METHODES.items():
        prevu = methode(historique, cible)
        for g in GROUPES:
            lignes.append({"methode": nom, "groupe": g, "reel": mois.loc[cible, g], "prevu": prevu[g]})
test = pd.DataFrame(lignes)
test["erreur %"] = (test["prevu"] - test["reel"]).abs() / test["reel"] * 100

mape = test.pivot_table(index="groupe", columns="methode", values="erreur %", aggfunc="mean")[list(METHODES)].round(1)

def conclure(gain):
    if gain >= GAIN_MINIMUM:
        return "Oui"
    if gain > 0:
        return "Gain marginal"
    return "Non"

verdict = pd.DataFrame({
    "Meilleure naïve": mape[NAIVES].min(axis=1),
    "Notre meilleure": mape[MODELES].min(axis=1),
    "Modèle retenu": mape[MODELES].idxmin(axis=1),
})
verdict["Gain"] = (verdict["Meilleure naïve"] - verdict["Notre meilleure"]).round(1)
verdict["Prévisible ?"] = verdict["Gain"].apply(conclure)

print(f"Test sur {len(mois_testes)} mois : de {en_francais(mois_testes[0])} à {en_francais(mois_testes[-1])}")
verdict

Test sur 21 mois : de janvier 2018 à septembre 2019


,Meilleure naïve,Notre meilleure,Modèle retenu,Gain,Prévisible ?
groupe,,,,,
M01AB,9.9,7.5,Saison x niveau,2.4,Oui
M01AE,12.5,12.7,Saison x niveau,-0.2,Non
N02BA,10.8,10.4,Moyenne 3 derniers mois,0.4,Gain marginal
N02BE,20.4,13.0,Saison x niveau,7.4,Oui
N05B,14.4,11.5,Moyenne 3 derniers mois,2.9,Oui
N05C,39.8,33.8,Moyenne 3 derniers mois,6.0,Oui
R03,27.9,30.0,Saison x niveau,-2.1,Non
R06,25.5,19.0,Saison x niveau,6.5,Oui


## Étape 4 : prévoir le mois suivant

Seulement pour les groupes jugés prévisibles. Pour les autres, le classeur affiche « Pas fiable ».

In [5]:
prevision = {}
for g in GROUPES:
    if verdict.loc[g, "Prévisible ?"] == "Oui":
        prevision[g] = round(METHODES[verdict.loc[g, "Modèle retenu"]](mois, mois_cible)[g])
    else:
        prevision[g] = "Pas fiable"

an_dernier = mois_cible - pd.DateOffset(years=1)
tableau_prevision = pd.DataFrame({
    f"Prévision {en_francais(mois_cible)}": prevision,
    f"Réel {en_francais(dernier_mois)}": mois.loc[dernier_mois].round(0),
    f"Réel {en_francais(an_dernier)}": mois.loc[an_dernier].round(0) if an_dernier in mois.index else None,
})
nb_oui = int((verdict["Prévisible ?"] == "Oui").sum())
print(f"Prévisible pour {nb_oui} groupes sur 8")
tableau_prevision

Prévisible pour 5 groupes sur 8


,Prévision octobre 2019,Réel septembre 2019,Réel octobre 2018
M01AB,161,161.0,145.0
M01AE,Pas fiable,111.0,129.0
N02BA,Pas fiable,94.0,76.0
N02BE,1307,984.0,1129.0
N05B,238,228.0,287.0
N05C,21,16.0,25.0
R03,Pas fiable,121.0,327.0
R06,75,109.0,77.0


## Étape 5 : ouvrir le classeur et remplacer l'onglet Données

L'onglet Données contient une ligne par jour. Les formules de l'onglet Ventes vont y chercher les chiffres.

In [6]:
wb = openpyxl.load_workbook(FICHIER_CLASSEUR)
donnees = wb["Données"]
ancienne_derniere_ligne = donnees.max_row

# On garde la mise en forme de la première ligne de données pour la recopier
modele = [copy(c._style) for c in donnees[2]]
formats = [c.number_format for c in donnees[2]]

donnees.delete_rows(2, ancienne_derniere_ligne)
for i, ligne in jour.iterrows():
    valeurs = [ligne["datum"].to_pydatetime()] + [ligne[g] for g in GROUPES] + [
        ligne["datum"].year, ligne["datum"].month, JOURS[ligne["datum"].weekday()]]
    for j, valeur in enumerate(valeurs):
        cellule = donnees.cell(row=i + 2, column=j + 1, value=valeur)
        cellule._style = copy(modele[j])
        cellule.number_format = formats[j]

derniere_ligne = len(jour) + 1
print(f"Onglet Données : {len(jour)} jours écrits (lignes 2 à {derniere_ligne})")

Onglet Données : 2106 jours écrits (lignes 2 à 2107)


## Étape 6 : adapter les formules et les dates du classeur

Les formules s'arrêtaient à l'ancienne dernière ligne : on les fait aller jusqu'à la nouvelle.
On met aussi à jour les dates affichées dans les textes.

In [7]:
nb_formules = 0
for feuille in wb.worksheets:
    for rangee in feuille.iter_rows():
        for c in rangee:
            if isinstance(c.value, str) and c.value.startswith("=") and f"${ancienne_derniere_ligne}" in c.value:
                c.value = re.sub(rf"\$(\w+)\${ancienne_derniere_ligne}\b", rf"$\g<1>${derniere_ligne}", c.value)
                nb_formules += 1

for nom in ["DateMin", "DateMax"]:
    if nom in wb.defined_names:
        wb.defined_names[nom].attr_text = re.sub(rf"\${ancienne_derniere_ligne}\b", f"${derniere_ligne}",
                                                 wb.defined_names[nom].attr_text)

periode = f"du {premier_jour:%d/%m/%Y} au {dernier_jour:%d/%m/%Y}"
ventes = wb["Ventes"]
ventes["A2"] = re.sub(r"du \d\d/\d\d/\d{4} au \d\d/\d\d/\d{4}", periode, ventes["A2"].value)
ventes["B5"] = (dernier_jour - pd.Timedelta(days=7)).to_pydatetime()
ventes["B6"] = dernier_jour.to_pydatetime()
wb["Questions"]["B14"] = f"{periode[0].upper()}{periode[1:]}. Une date en dehors est refusée."

print(f"{nb_formules} formules mises à jour, période : {periode}")

16 formules mises à jour, période : du 02/01/2014 au 08/10/2019


## Étape 7 : réécrire l'onglet Prévision

In [8]:
prev = wb["Prévision"]
periode_test = f"{en_francais(mois_testes[0])} à {en_francais(mois_testes[-1])}"
non = [f"{NOMS[g]} ({g})" for g in GROUPES if verdict.loc[g, "Prévisible ?"] == "Non"]
marginal = [f"{NOMS[g]} ({g})" for g in GROUPES if verdict.loc[g, "Prévisible ?"] == "Gain marginal"]

reponse = f"{'Oui' if nb_oui else 'Non'}, pour {nb_oui} groupes sur 8"
if verdict.loc["N02BE", "Prévisible ?"] == "Oui":
    reponse += ", dont le paracétamol qui représente la moitié des ventes"
reponse += "."
if non:
    reponse += " Non pour : " + ", ".join(non) + "."
if marginal:
    reponse += " Gain trop faible pour : " + ", ".join(marginal) + "."

prev["A2"] = (f"Question du manager : prévoir, pour chaque groupe, le mois qui suit le dernier mois complet "
              f"({en_francais(dernier_mois)}), donc {en_francais(mois_cible)}.")
prev["A6"] = reponse
prev["A9"] = (f"1. On se replace chaque mois de {periode_test} ({len(mois_testes)} mois) et on prévoit le mois suivant "
              "avec uniquement les ventes connues à ce moment-là. Puis on compare au vrai chiffre.")

for i, g in enumerate(GROUPES):
    r = 16 + i
    for j, m in enumerate(METHODES):
        prev.cell(r, 3 + j).value = mape.loc[g, m] / 100
    prev.cell(r, 7).value = verdict.loc[g, "Meilleure naïve"] / 100
    prev.cell(r, 8).value = verdict.loc[g, "Notre meilleure"] / 100
    prev.cell(r, 9).value = verdict.loc[g, "Gain"]
    prev.cell(r, 10).value = verdict.loc[g, "Prévisible ?"]

couleurs = {"Oui": "FFC6EFCE", "Non": "FFFFC7CE", "Gain marginal": "FFFFEB9C"}
for i, g in enumerate(GROUPES):
    prev.cell(16 + i, 10).fill = PatternFill("solid", fgColor=couleurs[verdict.loc[g, "Prévisible ?"]])

prev["A25"] = f"PRÉVISION POUR {en_francais(mois_cible).upper()}"
prev["C26"] = f"Prévision {mois_cible:%m/%Y}"
prev["D26"] = f"Réel {dernier_mois:%m/%Y}"
prev["E26"] = f"Réel {an_dernier:%m/%Y}"
for i, g in enumerate(GROUPES):
    r = 27 + i
    c = prev.cell(r, 3)
    c.value = prevision[g]
    c.fill = PatternFill("solid", fgColor=couleurs["Oui" if prevision[g] != "Pas fiable" else "Gain marginal"])
    prev.cell(r, 4).value = int(round(mois.loc[dernier_mois, g]))
    prev.cell(r, 5).value = int(round(mois.loc[an_dernier, g])) if an_dernier in mois.index else None
    if prevision[g] != "Pas fiable":
        prev.cell(r, 6).value = (f"{verdict.loc[g, 'Modèle retenu']} (erreur moyenne "
                                 f"{verdict.loc[g, 'Notre meilleure']:.1f} % sur {len(mois_testes)} mois)")
    else:
        prev.cell(r, 6).value = "Aucune méthode ne bat la prévision naïve : se fier au calendrier des saisons"

prev["A36"] = (f"Mise à jour automatique par classeur.ipynb avec l'export du {dernier_jour:%d/%m/%Y}. "
               f"{en_francais(mois_cible).capitalize()} ne peut pas encore servir à vérifier la prévision : "
               f"il n'est pas terminé dans l'export.")
print(reponse)

Oui, pour 5 groupes sur 8, dont le paracétamol qui représente la moitié des ventes. Non pour : Anti-inflammatoires type ibuprofène (M01AE), Asthme et BPCO (R03). Gain trop faible pour : Aspirine et dérivés (N02BA).


## Étape 8 : mettre à jour les lignes « prévision » de l'onglet Synthèse

In [9]:
synthese = wb["Synthèse"]
cible_court = f"{mois_cible:%m/%Y}"
para = verdict.loc["N02BE"]
synthese["B25"] = (f"Peut-on prévoir les ventes du mois suivant ({cible_court}) ? Oui pour {nb_oui} groupes sur 8"
                   + (", dont le paracétamol (la moitié des ventes)." if para["Prévisible ?"] == "Oui" else "."))
synthese["C25"] = pourcent(para["Notre meilleure"])
synthese["D25"] = f"Erreur moyenne de notre prévision, paracétamol ({len(mois_testes)} mois testés)"
synthese["E25"] = pourcent(para["Meilleure naïve"])
synthese["G25"] = f"Oui : testé sur {len(mois_testes)} mois non utilisés pour construire la prévision"
synthese["B56"] = f"Peut-on prévoir le mois suivant ({cible_court}) ?"
synthese["C56"] = f"{'OUI' if nb_oui else 'NON'}, pour {nb_oui} groupes sur 8"
synthese["D56"] = f"Testé sur {len(mois_testes)} mois. Voir l'onglet Prévision."
synthese["C69"] = f"Peut-on prévoir le mois suivant ? Le test, et la prévision {cible_court}"
print("Synthèse mise à jour")

Synthèse mise à jour


## Étape 9 : recalculer l'onglet Statistiques

Les mêmes calculs que `statistiques.py`, écrits directement dans le classeur :
par groupe, indice saisonnier, moyenne de chaque mois, jour de la semaine, heure, année, et chaque mois complet.

In [10]:
stats = wb["Statistiques"]
style_titre, style_entete = copy(stats["A5"]._style), copy(stats["A6"]._style)
style_premiere_col, style_valeur = copy(stats["A7"]._style), copy(stats["C7"]._style)
stats.delete_rows(5, stats.max_row)

ventes_jour = jour.set_index("datum")[GROUPES]
horaire = pd.read_csv(FICHIER_HORAIRE)
horaire["datum"] = pd.to_datetime(horaire["datum"], format="%m/%d/%Y %H:%M")
par_heure = horaire.groupby(horaire["datum"].dt.hour)[GROUPES].mean().round(3)
par_heure = par_heure[horaire.groupby(horaire["datum"].dt.hour)[GROUPES].sum().sum(axis=1) > 0]
mois_moyen = mois.groupby(mois.index.month).mean()

tableaux = []
tableaux.append(("PAR GROUPE (VENTES PAR JOUR, PUIS PAR MOIS COMPLET)",
    ["Groupe", "Nom", "Total", "Moyenne par jour", "Médiane par jour", "Écart-type par jour",
     "Coefficient de variation", "Maximum sur un jour", "Part des jours sans vente",
     "Moyenne par mois complet", "Mois le plus bas", "Mois le plus haut"],
    [[g, NOMS[g], round(ventes_jour[g].sum(), 1), round(ventes_jour[g].mean(), 2), round(ventes_jour[g].median(), 2),
      round(ventes_jour[g].std(), 2), round(ventes_jour[g].std() / ventes_jour[g].mean(), 2),
      round(ventes_jour[g].max(), 1), round((ventes_jour[g] == 0).mean(), 3),
      round(mois[g].mean(), 1), round(mois[g].min(), 1), round(mois[g].max(), 1)] for g in GROUPES]))
noms_mois = [m.capitalize() for m in MOIS_FR]
tableaux.append(("INDICE SAISONNIER (1 = MOIS MOYEN DU GROUPE)", ["Mois"] + GROUPES,
    [[noms_mois[m - 1]] + (mois_moyen.loc[m] / mois.mean()).round(2).tolist() for m in mois_moyen.index]))
tableaux.append(("VENTES MOYENNES DE CHAQUE MOIS DE L'ANNÉE", ["Mois"] + GROUPES,
    [[noms_mois[m - 1]] + mois_moyen.loc[m].round(1).tolist() for m in mois_moyen.index]))
semaine = ventes_jour.groupby(ventes_jour.index.weekday).mean().round(2)
tableaux.append(("VENTES MOYENNES PAR JOUR DE LA SEMAINE", ["Jour"] + GROUPES,
    [[JOURS[j]] + semaine.loc[j].tolist() for j in semaine.index]))
tableaux.append(("VENTES MOYENNES PAR HEURE", ["Heure"] + GROUPES,
    [[f"{h}h"] + par_heure.loc[h].tolist() for h in par_heure.index]))
annees = ventes_jour.groupby(ventes_jour.index.year)
tableaux.append(("VENTES MOYENNES PAR JOUR, ANNÉE PAR ANNÉE", ["Année", "Jours de données"] + GROUPES,
    [[a, int(annees.size()[a])] + annees.mean().round(2).loc[a].tolist() for a in annees.size().index]))
tableaux.append(("VENTES DE CHAQUE MOIS COMPLET, RECALCULÉES À PARTIR DES JOURS", ["Mois complet"] + GROUPES,
    [[f"{d:%Y-%m}"] + mois.loc[d].round(2).tolist() for d in mois.index]))

r = 5
for titre, entetes, lignes in tableaux:
    stats.cell(r, 1, titre)._style = copy(style_titre)
    stats.row_dimensions[r].height = 24
    r += 1
    for j, h in enumerate(entetes):
        stats.cell(r, j + 1, h)._style = copy(style_entete)
    stats.row_dimensions[r].height = 34.65 if len(entetes) > 10 else 20.25
    r += 1
    for ligne in lignes:
        for j, v in enumerate(ligne):
            v = v.item() if hasattr(v, "item") else v
            stats.cell(r, j + 1, v)._style = copy(style_premiere_col if j == 0 else style_valeur)
        stats.row_dimensions[r].height = 20.25
        r += 1
    r += 2

print(f"Onglet Statistiques réécrit : {len(tableaux)} tableaux, {len(mois)} mois complets")

Onglet Statistiques réécrit : 7 tableaux, 68 mois complets


## Étape 10 : enregistrer et vérifier

Excel recalcule toutes les formules à l'ouverture du fichier.

In [11]:
wb.calculation.fullCalcOnLoad = True
wb.save(FICHIER_CLASSEUR)

verif = openpyxl.load_workbook(FICHIER_CLASSEUR)
print("Classeur enregistré :", FICHIER_CLASSEUR)
print("Onglets :", ", ".join(verif.sheetnames))
print("Dernière date dans Données :", verif["Données"].cell(verif["Données"].max_row, 1).value.strftime("%d/%m/%Y"))
print(verif["Prévision"]["A25"].value)
for r in range(27, 35):
    print(f"  {verif['Prévision'].cell(r, 1).value:<40} {verif['Prévision'].cell(r, 3).value}")
print()
print("À relire à la main (textes écrits par l'équipe, pas recalculés) :")
print("  onglets Calendrier, Conseils, État des données, et les autres lignes de Synthèse.")

Classeur enregistré : excel/Pharma_Analyse.xlsx
Onglets : Synthèse, Ventes, Calendrier, Conseils, Prévision, Questions, Médicaments, Méthode, Statistiques, État des données, Données
Dernière date dans Données : 08/10/2019
PRÉVISION POUR OCTOBRE 2019
  Anti-inflammatoires type diclofénac      161
  Anti-inflammatoires type ibuprofène      Pas fiable
  Aspirine et dérivés                      Pas fiable
  Paracétamol                              1307
  Anxiolytiques                            238
  Somnifères et sédatifs                   21
  Asthme et BPCO                           Pas fiable
  Antihistaminiques (allergies)            75

À relire à la main (textes écrits par l'équipe, pas recalculés) :
  onglets Calendrier, Conseils, État des données, et les autres lignes de Synthèse.
